# RetailMind AI — Day 5: Keras neural network (ANN) vs the classical champion

**Scenario.** RetailMind AI is built one increment per day in the same repo. Days 1-4 built the churn pipeline, compared and tuned classical models, tried new features and PCA, and ran a fair 12-model comparison (winner: **Bagging**, F1 0.6401, ROC-AUC 0.8462).

**Day 5 objective.** Build a Keras ANN for churn and benchmark it head-to-head against that classical champion, then record honestly whether depth helped.

**How it connects to earlier days**
- Same customers and the same train/test split (`src/data.py`), so the scores are directly comparable.
- Same data-preparation step (`src/pipeline.py`: scaling + one-hot), reused as the ANN's input step.
- Same scoring code (`src/evaluate.py`, threshold 0.50), now usable for any model that outputs probabilities.
- Same class-imbalance idea as Day 2 (`class_weight`), applied to the network as a class weight.

**Business value.** The question "should we move to deep learning for churn?" now has a measured answer instead of an opinion, before any money is spent on GPUs or on serving TensorFlow.

In [1]:
# Work from the project root so the `src` package can be imported
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("project folder:", os.path.basename(ROOT))

project folder: day3-4


## 1. Warm-up: a tiny network on MNIST
MNIST is a set of handwritten digits (28x28 pixel images). We flatten each image into 784 numbers (scaled 0-1) and train one hidden layer of 64 ReLU neurons with a 10-way softmax output. It is only here to see layers, activations and a training loop working before we use them on churn.

In [2]:
from src.train_day5 import mnist_warm_up
mnist_accuracy = mnist_warm_up()

Step 1: warm-up on MNIST (images of the digits 0 to 9)


  after 3 rounds (epochs) the network gets 89.6% of unseen digits right


## 2. Train the churn networks and benchmark them
`main()` does the whole increment:
1. prepares the same 45 columns (scaled numbers + one-hot text) from the same training customers
2. builds a **shallow** network (1 hidden layer of 16) and a **deeper** one (64-32-16 with 30% dropout), both ending in one sigmoid neuron
3. trains each with **early stopping** on validation loss (patience 10, best weights restored) and a class weight for the minority churners
4. picks the network by **validation loss** (never by the test customers)
5. scores it against the Phase 1 champion on the same 1,409 test customers, then saves the model, its preparation step and the results

In [3]:
from src.train_day5 import main
main()

Step 1: warm-up on MNIST (images of the digits 0 to 9)


  after 3 rounds (epochs) the network gets 89.6% of unseen digits right

Step 2: build and train the churn networks
  input columns after preparation: 45


  Shallow ANN (1 hidden layer of 16): 753 weights, stopped after 25 rounds, best round 15, best validation loss 0.5251


  Deeper ANN (3 hidden layers: 64, 32, 16): 5569 weights, stopped after 17 rounds, best round 7, best validation loss 0.5101

Step 3: choose the network
  lower validation loss wins -> Deeper ANN (3 hidden layers: 64, 32, 16)

Step 4: test customers, ANN vs the best classical model
  Model                                         Recall  Precision      F1  ROC-AUC
  Best classical model (Bagging, Phase 1)       0.7727     0.5463  0.6401   0.8462
  Shallow ANN (1 hidden layer of 16)            0.7834      0.506  0.6149   0.8402
  Deeper ANN (3 hidden layers: 64, 32, 16)       0.746     0.5186  0.6118   0.8389  <- chosen ANN
  -> Depth did not help: the ANN F1 is 0.0283 lower than the classical model.
  -> shallow vs deeper network: F1 0.6149 vs 0.6118



Saved network     -> models/churn_ann.keras
Saved preparation -> models/churn_ann_prepare.joblib
Saved results     -> models/day5_results.json


## 3. The benchmark table

In [4]:
import json
import pandas as pd

with open("models/day5_results.json") as f:
    day5 = json.load(f)

rows = {"Best classical model (Bagging, Phase 1)": day5["classical_champion_test_scores"]}
for name, info in day5["networks"].items():
    rows[name] = info["test_scores"]

table = pd.DataFrame(rows).T[["recall", "precision", "f1", "roc_auc"]]
display(table)
print("Chosen network (lowest validation loss):", day5["chosen_network"])
print("F1 gap vs classical:", day5["f1_gap_vs_classical"], "| ROC-AUC gap:", day5["roc_auc_gap_vs_classical"])
print("Verdict:", day5["verdict"])

,recall,precision,f1,roc_auc
"Best classical model (Bagging, Phase 1)",0.7727,0.5463,0.6401,0.8462
Shallow ANN (1 hidden layer of 16),0.7834,0.506,0.6149,0.8402
"Deeper ANN (3 hidden layers: 64, 32, 16)",0.746,0.5186,0.6118,0.8389


Chosen network (lowest validation loss): Deeper ANN (3 hidden layers: 64, 32, 16)
F1 gap vs classical: -0.0283 | ROC-AUC gap: -0.0073
Verdict: Depth did not help: the ANN F1 is 0.0283 lower than the classical model.


## 4. Training behaviour: did early stopping matter?
Rounds trained vs the best round shows how quickly each network started to memorise the training customers.

In [5]:
training = pd.DataFrame({name: {k: v for k, v in info.items() if k != "test_scores"}
                         for name, info in day5["networks"].items()}).T
display(training)

,parameters,rounds_trained,best_round,best_validation_loss,final_training_loss
Shallow ANN (1 hidden layer of 16),753.0,25.0,15.0,0.5251,0.6719
"Deeper ANN (3 hidden layers: 64, 32, 16)",5569.0,17.0,7.0,0.5101,0.7186


## 5. Serving: score customers with the saved network
The saved `.keras` model and its preparation step are loaded by `src/ann_service.py`. For comparison, the classical service (`src/service.py`) scores the same two sample customers.

In [6]:
from src.ann_service import predict_with_ann
from src.service import SAMPLE_CUSTOMERS, predict

print("Neural network:")
for answer in predict_with_ann(SAMPLE_CUSTOMERS):
    print(" ", answer)

print("Classical champion (service.py):")
for answer in predict(SAMPLE_CUSTOMERS):
    print(" ", answer)

Neural network:
  {'customer': 1, 'churn_probability': 0.753, 'churn_prediction': 'Yes', 'risk_tier': 'high'}
  {'customer': 2, 'churn_probability': 0.1391, 'churn_prediction': 'No', 'risk_tier': 'low'}
Classical champion (service.py):
  {'churn_probability': 0.7831, 'churn_prediction': 'Yes', 'risk_tier': 'high'}
  {'churn_probability': 0.2126, 'churn_prediction': 'No', 'risk_tier': 'low'}


## 6. Conclusion

**Did depth help? No, not on this data.** The chosen ANN's F1 is about 0.028 below the classical champion's and its ROC-AUC about 0.007 below. The deeper network was also not better than the shallow one on the test customers, although it had the lower validation loss. A table of about 7,000 customers with 19 clean columns gives a neural network little to discover that a tree-based model has not already found.

**When to use which**
| Situation | Better choice |
| --- | --- |
| Tabular customer data like this | Classical tree-based or linear model (matched or beat the ANN, trains in seconds, small to deploy, explainable) |
| Images, audio, free text, very large data | Neural network (it learns its own features). Day 6 moves to product photos with a CNN |
| Small data | Classical model (networks memorise without early stopping and dropout) |

**Decision.** The classical champion stays the production model; the ANN is kept as a measured benchmark (`models/churn_ann.keras`).

**Open blockers / notes for doubt-clearing**
- These ANN numbers come from one training run with a fixed seed (42). Gaps of a few hundredths between the two networks are within what another seed could change.
- Serving the ANN needs TensorFlow, which supports Python 3.9-3.12; the classical model needs only scikit-learn.
- Days 3-5 are on stacked, still-unmerged branches in the repo (`day3-4-tuning-and-ensembles`, then `day5-keras-ann`).